<a href="https://colab.research.google.com/github/JamshedAli18/LLm-finetuning/blob/main/LoraFit_SmolLM2_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install everything**

In [ ]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.0 MB/s eta 0:00:00


**Pick your dataset**

*   The best beginner dataset is Alpaca (tatsu-lab/alpaca).
*   It has 52K instruction-following examples already in the exact format SmolLM2 expects.



In [ ]:
from datasets import load_dataset

# Load the Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
# Preview what it looks like
dataset[3]

{'instruction': 'How can we reduce air pollution?',
 'input': '',
 'output': 'There are a number of ways to reduce air pollution, such as shifting to renewable energy sources, encouraging the use of public transportation, prohibiting the burning of fossil fuels, implementing policies to reduce emissions from industrial sources, and implementing vehicle emissions standards. Additionally, individuals can do their part to reduce air pollution by reducing car use, avoiding burning materials such as wood, and changing to energy efficient appliances.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nHow can we reduce air pollution?\n\n### Response:\nThere are a number of ways to reduce air pollution, such as shifting to renewable energy sources, encouraging the use of public transportation, prohibiting the burning of fossil fuels, implementing policies to reduce emissions from industrial sources, and imp

**Format into SmolLM2's chat template**

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def format_alpaca_to_chat(example):
    """Convert an Alpaca row into SmolLM2's ChatML format."""
    # Build the user message
    if example["input"]:
        user_msg = f"{example['instruction']}\n\n{example['input']}"
    else:
        user_msg = example["instruction"]

    # Apply the model's native chat template
    messages = [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": example["output"]},
    ]
    # tokenizer.apply_chat_template renders it to a string
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,          # return string, not token ids
        add_generation_prompt=False
    )
    return {"text": text}

# Apply to the full dataset
dataset = dataset.map(format_alpaca_to_chat)

# See what the formatted text looks like
print(dataset[0]["text"])

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Give three tips for staying healthy.<|im_end|>
<|im_start|>assistant
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.<|im_end|>



In [ ]:
dataset = dataset.select(range(1000)).train_test_split(test_size=0.1)
print(f"Train: {len(dataset['train'])}  |  Val: {len(dataset['test'])}")

Train: 900  |  Val: 100


**Load model in 4-bit (QLoRA)**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config (the "Q" in QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16, store in 4-bit
    bnb_4bit_use_double_quant=True,       # double quantization saves ~0.4 bits/param
)

# Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",         # puts it on GPU automatically
    trust_remote_code=True,
)

# Prepare frozen layers for gradient checkpointing (saves memory)
model = prepare_model_for_kbit_training(model)

# LoRA config — which layers to add adapters to
lora_config = LoraConfig(
    r=16,                      # rank: 16 is a good default for 135M model
    lora_alpha=32,             # scaling factor α (rule of thumb: 2×r)
    target_modules=[           # which weight matrices to add adapters to
        "q_proj", "k_proj",    # attention query and key
        "v_proj", "o_proj",    # attention value and output
        "gate_proj", "up_proj", "down_proj",  # MLP layers
    ],
    lora_dropout=0.05,         # small dropout to prevent overfitting
    bias="none",               # don't train bias terms
    task_type="CAUSAL_LM",     # language modeling task
)

# Wrap the model with LoRA adapters
model = get_peft_model(model, lora_config)

# See how many params are trainable
model.print_trainable_parameters()
# Output: trainable params: ~1.3M || all params: ~136M || trainable: ~0.97%

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039


**Train with SFTTrainer**

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./smollm2-alpaca-lora",

    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,

    fp16=False,   # <-- turned off
    bf16=True,    # <-- turned on

    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=57, training_loss=2.142785908883078, metrics={'train_runtime': 162.8022, 'train_samples_per_second': 5.528, 'train_steps_per_second': 0.35, 'total_flos': 95676040111104.0, 'train_loss': 2.142785908883078})

 **Save the LoRA adapter**

In [ ]:
# Save just the LoRA adapter weights (very small — ~10MB)
trainer.save_model("./smollm2-alpaca-lora/final-adapter")
tokenizer.save_pretrained("./smollm2-alpaca-lora/final-adapter")

print("Adapter saved!")

Adapter saved!


# **Inference**

In [ ]:
# Disable gradient checkpointing for inference
model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()

def chat(instruction):
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!
print(chat("Give me 3 tips for staying productive."))
print("---")
print(chat("Explain black holes simply."))
print("---")
print(chat("Write a short poem about rain."))

1. Set clear goals and prioritize tasks to achieve them.

2. Take regular breaks throughout the day to help refresh your mind and reduce stress.

3. Eliminate distractions when you're not focused on work or an activity.

4. Maintain a consistent routine with meals, sleep, and physical exercise.
---
Ablack hole is a region in space where gravity has become so strong that nothing can escape from it due to the force of gravity being too huge and powerful. Since matter doesn't fall back into its original place after falling through, matter gets pulled towards very large radii as well as massive masses, causing the most massive objects will be torn apart or collide since they have no chance at all if there were no black-holes in their vicinity! So these regions come with an end because you aren't able to go near them.
---
Rain is the gentle drizzle that soaks into every corner of our lives; it brings with it warmth and light to all who dwell there; but also its fury can be fierce when we le

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!")

True
Tesla T4
